# 梯度下降优化算法全景解析：核心思想与演进逻辑

## 核心视角

本文的讨论视角是：**从某个初始参数点出发，使用某种迭代算法对固定目标函数（损失函数）进行确定性寻优**。

在这个视角下，所有算法回答的都是同一个问题：**"当前在参数点 θₜ，已知该点的梯度信息 g(θₜ)，下一步应该走到 θₜ₊₁ ？"**

本文讨论的核心是**参数更新策略**——拿到梯度后，如何构造下一步的移动方向和步长。

> **重要说明**：
> 1. 本文只讨论**一阶梯度方法（仅使用梯度，不使用Hessian二阶信息）**，排除拟牛顿法L-BFGS。
> 2. 本文采用**确定性梯度**假设（即全量梯度下降，GD）。Mini-batch 引入的采样噪声属于数据采样层面，不在本文讨论范围内。
> 3. 在确定性框架下，**任何一阶梯度方法都无法保证逃离局部极小值**——它们都只能收敛到初始点所在吸引域内的某个局部极小值。这一根本限制由优化问题的非凸性决定，而非算法设计所能突破。

> **核心理念**：本文呈现的是算法设计的演进逻辑——每一代算法针对特定问题提出解决方案，同时引入新的权衡（trade-off）。不存在"更好"或"更先进"的算法，只有**更适合特定场景**的算法。

# 第一部分：评估优化器性能的完整维度

在深入每个算法之前，先建立统一的评估框架。一个优化器在特定任务上的表现，可以从以下**6个核心维度**来衡量。后续每个算法的分析都将围绕这些维度展开。

## 1.1 收敛速度（Convergence Speed）

最直观的指标：达到目标精度或损失值所需的迭代次数。
- **关注点**：前期快速下降 vs 后期精细收敛。
- **典型对比**：GD 在平坦区域推进极慢；Heavy Ball 利用惯性加速；Adam 自适应放大步长。

## 1.2 山谷震荡抑制（Oscillation Suppression in Narrow Valleys）

损失曲面一个方向很缓、一个方向很陡时（即条件数大），梯度方向会反复横跳，路径呈锯齿状。
- **核心问题**：算法能否抑制垂直方向的震荡，加速水平方向的推进。
- **典型对比**：GD 在此类地形中震荡严重、收敛极慢；Heavy Ball 通过历史动量有效抵消垂直震荡。

## 1.3 逃离鞍点（Saddle Point Escape）

鞍点处梯度为零，但Hessian矩阵有正有负，存在方向信息——某些方向是上升的（正曲率），某些方向是下降的（负曲率）。
- **核心问题**：算法能否利用曲率的方向信息，快速选择正确的下降方向离开鞍点。
- **关键区别**：与平坦区域不同，鞍点处**有方向信息**（Hessian有正有负），算法可利用动量方向或自适应步长**判断往哪走**。
- **典型对比**：GD 在鞍点处梯度为零即停滞；Heavy Ball 可凭借惯性冲过去；Adam 可通过自适应机制放大信号加速离开。

## 1.4 超参数鲁棒性（Hyperparameter Robustness）

算法对学习率等超参数的敏感程度，直接影响调参成本。
- **表现**：自适应算法（如 Adam）对学习率的容忍范围较宽（如 1e-4 ~ 1e-2 均可收敛）；而 GD / Heavy Ball 对学习率极其敏感。
- **意义**：鲁棒性差的算法，调参时间可能超过训练时间本身。

## 1.5 内存与计算开销（Memory & Compute Overhead）

算法额外维护的状态变量占用的存储和计算资源。
- **显存占用**：
  - **GD**：不维护任何额外状态。
  - **Heavy Ball / NAG**：维护一个动量向量 v。
  - **Adam / AdamW**：维护一阶动量 m 和二阶动量 v，参数量翻倍。
  - **Muon**：维护一个动量缓冲区，约 Adam 的一半。
- **计算量**：自适应算法需要额外的乘加运算和开根号操作。

## 1.6 尺度不变性（Scale Invariance）

指算法对输入特征或参数量级的敏感程度。
- **GD / Heavy Ball / NAG**：对特征尺度极其敏感，需要精细的归一化处理。
- **自适应算法（Adagrad / RMSprop / Adam）**：由于为每个参数独立调整学习率，天然具有尺度不变性。
- **Rprop**：只使用梯度符号，不依赖梯度幅值，也具有尺度不变性。
- **Muon**：正交化使更新方向与梯度幅值解耦，对 2D 矩阵具有尺度不变性。

## 1.7 各算法六维能力速览（总览表）

> **关于"逃离局部极小值"**：在确定性梯度假设下，**任何一阶方法都无法保证逃离局部极小值**。所有算法最终都会收敛到初始点所在吸引域内的某个局部极小值。因此"逃离浅坑"不作为独立维度，而在 1.3 逃离鞍点中讨论——逃离鞍点与逃离局部极小值性质完全不同：鞍点处有方向信息可被利用，而局部极小值处所有方向都是上升的，算法在确定性框架下没有任何信息可以"判断往哪走"。

| 算法 | 收敛速度<br>Convergence Speed | 山谷震荡抑制<br>Oscillation Suppression | 逃离鞍点<br>Saddle Point Escape | 超参数鲁棒性<br>Hyperparameter Robustness | 显存开销<br>Memory Overhead | 尺度不变性<br>Scale Invariance |
|------|:---:|:---:|:---:|:---:|:---:|:---:|
| **GD** | 慢 | 差 | 差（梯度为零即停滞） | 差 | **极低** | 差 |
| **Heavy Ball（Momentum）** | 中快 | **好** | 中（惯性可冲过） | 中 | 低 | 差 |
| **NAG** | 中快 | **好**（优于Heavy Ball） | 中 | 中 | 低 | 差 |
| **Rprop** | 中 | **好** | 差（符号抖动时步长收缩） | 中 | 中 | **好** |
| **Adagrad** | 中（稀疏场景快） | 较好 | 中 | 中 | 中 | **好** |
| **RMSprop** | 较快 | 较好 | 较好 | 较好 | 中高 | **好** |
| **Adam** | **快** | **好** | **强**（动量+自适应） | **好** | **高** | **好** |
| **NAdam** | 快 | **好** | 强 | 好 | 高 | 好 |
| **RAdam** | 快（早期略慢） | 好 | 强 | **好**（早期更稳） | 高 | 好 |
| **AdamW** | **快** | **好** | **强** | **好** | **高** | **好** |
| **Muon** | **极快** | **好** | **强** | 中 | 中 | **好**（针对2D矩阵） |

**结论**：没有完美的算法。选择时需根据任务优先级进行**多维权衡**：
- **追求收敛速度** → Adam / AdamW
- **追求极简与可解释性** → GD
- **显存受限** → GD（但慢）或 Muon
- **需要尺度不变性** → 自适应类或 Rprop

# 第二部分：参数更新策略的演进（如何使用梯度信息）

## 问题框架

在每次迭代中，算法面对的是：
- 当前参数位置 θₜ
- 当前点的梯度信息 gₜ
- 历史梯度信息（如果有维护的话）

算法的任务：**决定下一步的移动 Δθₜ = θₜ₊₁ − θₜ**

以下按演进顺序介绍各个算法，每个算法附上六维评估。

## 2.1 最朴素的决策：梯度下降（Gradient Descent, GD）

### 迭代规则

$$
\Delta\theta_t = -\eta \cdot g(\theta_t)
$$

这是所有算法最原始的形态，也是**适用范围最广**的基线方法。

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 慢 | 在狭长山谷中锯齿震荡；在平坦区推进极慢 |
| **山谷震荡抑制** | **差** | 无任何机制抑制垂直方向震荡 |
| **逃离鞍点** | **差** | 梯度为零即完全停滞 |
| **超参数鲁棒性** | 差 | 对学习率极其敏感 |
| **显存开销** | **极低** | 不维护任何额外状态 |
| **尺度不变性** | 差 | 需精细归一化 |

### 这个决策的问题

1. 狭长山谷中锯齿震荡，收敛极慢
2. 所有参数共用同一个学习率，无法适应不同参数的尺度差异
3. 鞍点/平坦区梯度为零时完全停滞

### 适用场景

凸优化问题或曲率较为均衡的任务中，GD 依然表现良好，且超参数最少、最稳定。

## 2.2 针对山谷震荡的改进：重球法（Heavy Ball Method）

> 由 Polyak 于1964年提出。深度学习社区俗称 **Momentum（动量法）**。

### 针对的问题

GD 在狭长山谷中锯齿震荡，收敛缓慢。

### 核心思想

维护一个**累积的历史移动方向**作为惯性，让更新方向既参考当前梯度，也继承历史趋势。

**物理类比**：重球在损失曲面上滚动。球的质量带来惯性——即使当前坡度方向改变，球也会因之前的运动趋势而继续前进。这抑制了峡谷壁上的来回震荡。

### 迭代规则

$$
v_t = \beta v_{t-1} - \eta \cdot g(\theta_t)
$$
$$
\Delta\theta_t = v_t
$$

其中 $\beta \in [0.9, 0.99]$ 是动量衰减系数。

### 数学本质

重球法引入了**二阶动力学**（虽然仍是一阶梯度方法）：

$$
\theta_{t+1} = \theta_t - \eta g(\theta_t) + \beta(\theta_t - \theta_{t-1})
$$

即：新更新 = 当前梯度项 + 上一次更新方向的衰减延续。

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 中快 | 抑制震荡，加速推进 |
| **山谷震荡抑制** | **好** | 历史动量抵消垂直震荡 |
| **逃离鞍点** | **中** | 惯性可在梯度为零时推动参数前进 |
| **超参数鲁棒性** | 中 | 需调 η 和 β，但 β 在0.9-0.99区间较稳定 |
| **显存开销** | 低 | 额外维护一个动量向量 v |
| **尺度不变性** | 差 | 仍为统一学习率 |

### 权衡

动量有"滞后"效应：急转弯时会冲过头，导致超调震荡。

## 2.3 针对动量滞后的改进：Nesterov Accelerated Gradient（NAG）

> 由 Nesterov 于1983年提出。深度学习社区俗称 **Nesterov Momentum**。

### 针对的问题

Heavy Ball 在需要急转弯时，由于历史惯性积累，会冲过头导致超调。

### 核心思想

既然历史动量已经让参数有向前冲的趋势，那就在这个"预判位置"计算梯度，而不是在原地算。

### 迭代规则

$$
v_t = \beta v_{t-1} - \eta \cdot g(\theta_t + \beta v_{t-1})
$$
$$
\Delta\theta_t = v_t
$$

关键差异：梯度计算的位置从 θₜ 变成了 **θₜ + βvₜ₋₁**（即按历史惯性往前看一步的位置）。

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 中快（略优于Heavy Ball） | 提前看到梯度变化，减少超调震荡 |
| **山谷震荡抑制** | **好**（优于Heavy Ball） | 预判机制使震荡抑制更精准 |
| **逃离鞍点** | **中** | 与Heavy Ball类似 |
| **超参数鲁棒性** | 中 | 与Heavy Ball类似 |
| **显存开销** | 低 | 额外维护一个动量向量 v |
| **尺度不变性** | 差 | 仍为统一学习率 |

### 权衡

仍然所有参数共用同一个学习率。对于梯度尺度差异巨大的参数，统一的学习率无法同时满足两者的需求。

## 2.4 针对统一学习率的改进：Rprop

> 由 Braun 和 Riedmiller 于1993年提出。

### 针对的问题

所有参数共用同一个学习率，无法适应不同参数的梯度尺度差异。

### 核心思想

为**每个参数单独维护一个步长**。更新时只使用梯度的**符号**来决定增加还是减少步长，而不使用梯度的大小。

### 迭代规则

对每个参数 $i$：

$$
\text{if } g_{t,i} \cdot g_{t-1,i} > 0 \quad \rightarrow \quad \Delta_i \leftarrow \min(\Delta_i \cdot \eta^+, \Delta_{max})
$$
$$
\text{if } g_{t,i} \cdot g_{t-1,i} < 0 \quad \rightarrow \quad \Delta_i \leftarrow \max(\Delta_i \cdot \eta^-, \Delta_{min})
$$
$$
\Delta\theta_{t,i} = -\text{sign}(g_{t,i}) \cdot \Delta_i
$$

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 中 | 每个参数独立步长，适应性强；但梯度幅值信息被丢弃 |
| **山谷震荡抑制** | **好** | 独立步长可分别适应缓坡和陡坡方向 |
| **逃离鞍点** | **差** | 鞍点处梯度符号频繁抖动，步长异常收缩，陷入停滞 |
| **超参数鲁棒性** | 中 | 需要调增/减因子和步长上下界 |
| **显存开销** | 中 | 为每个参数维护步长状态 |
| **尺度不变性** | 好 | 每个参数独立步长，不依赖绝对尺度 |

### 权衡

Rprop**依赖确定性梯度**。当梯度带有噪声时，梯度的符号会频繁随机翻转，导致步长异常收缩。因此Rprop**仅适用于确定性梯度场景**。

## 2.5 针对噪声下符号不可靠的改进：Adagrad

> 由 Duchi 等人于2011年提出。

### 针对的问题

Rprop 依赖梯度符号，在噪声场景下符号频繁抖动，步长异常收缩。

### 核心思想

不再依赖梯度符号，而是用**梯度的大小**来调整步长：梯度大的参数步长小，梯度小的参数步长大。

### 迭代规则

对每个参数 $i$：

$$
r_{t,i} = r_{t-1,i} + g_{t,i}^2
$$
$$
\Delta\theta_{t,i} = -\frac{\eta}{\sqrt{r_{t,i} + \epsilon}} \cdot g_{t,i}
$$

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 稀疏场景**极快** | 稀疏参数累积平方和小 → 学习率大 → 快速更新 |
| **山谷震荡抑制** | **较好** | 自适应步长可在一定程度上缓解震荡 |
| **逃离鞍点** | **中** | 对噪声有一定容忍度，但对鞍点无特殊处理 |
| **超参数鲁棒性** | 较好 | 对学习率有一定包容度 |
| **显存开销** | 中 | 需维护每个参数的梯度平方累加和 r |
| **尺度不变性** | **好** | 每个参数独立缩放 |

### 权衡

$r_{t,i}$ 是历史梯度平方的**累加和**，单调递增。训练后期，所有参数的学习率都趋近于0，训练**强制停滞**。

## 2.6 针对训练停滞的改进：RMSprop / Adadelta

> RMSprop 由 Hinton 于2012年提出（未正式发表）；Adadelta 由 Zeiler 于2012年提出。

### 针对的问题

Adagrad 中历史梯度平方的累加和单调递增，导致训练后期学习率趋近于0。

### 核心思想

用**指数移动平均（EMA）** 替代全量累加，让近期的梯度有更大权重，远期的逐渐遗忘。

### 迭代规则（RMSprop）

对每个参数 $i$：

$$
r_{t,i} = \rho r_{t-1,i} + (1-\rho) g_{t,i}^2
$$
$$
\Delta\theta_{t,i} = -\frac{\eta}{\sqrt{r_{t,i} + \epsilon}} \cdot g_{t,i}
$$

其中 $\rho \in [0.9, 0.999]$ 是衰减率。

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 较快 | 学习率持续适应地形，不会衰减到0 |
| **山谷震荡抑制** | **较好** | 自适应步长持续有效 |
| **逃离鞍点** | **较好** | EMA 保持步长活性，不会像 Adagrad 那样趋近0 |
| **超参数鲁棒性** | 较好 | 对 ρ 和 η 的容忍度较高 |
| **显存开销** | 中高 | 需维护每个参数的 EMA 平方和 r |
| **尺度不变性** | **好** | 每个参数独立缩放 |

### 权衡

只有二阶矩自适应（用梯度平方缩放学习率），缺少一阶动量加速（像 Heavy Ball 那样的惯性累积）。在平滑地形上收敛偏慢。

## 2.7 融合两种策略：Adam

> 由 Kingma 和 Ba 于2014年提出。

### 设计思路

将 Heavy Ball（一阶动量加速）和 RMSprop（二阶矩自适应步长）结合起来，同时获得两种能力。

### 迭代规则

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{（一阶动量，类似Heavy Ball）}
$$
$$
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{（二阶矩，类似RMSprop）}
$$
$$
\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{（偏差修正）}
$$
$$
\Delta\theta_t = -\eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | **快** | 动量加速 + 自适应步长，双管齐下 |
| **山谷震荡抑制** | **好** | 动量机制有效抑制震荡 |
| **逃离鞍点** | **强** | 动量 + 自适应步长组合，方向判断能力最强 |
| **超参数鲁棒性** | **好** | 对学习率容忍范围宽（1e-4 ~ 1e-2 均可收敛） |
| **显存开销** | **高** | 需维护 m 和 v 两个状态，参数量翻倍 |
| **尺度不变性** | **好** | 每个参数独立自适应 |

### 权衡

- 普通动量的滞后超调问题依然存在 → NAdam
- 训练初期二阶矩估计的方差可能很大 → RAdam
- 泛化短板（在非凸问题中可能收敛到尖锐极小值）

## 2.8 Adam 的变体

### NAdam：融入 Nesterov 预判

> 由 Dozat 于2016年提出。

**设计思路**：在 Adam 的一阶动量上叠加 Nesterov 的"提前看"机制。

**针对的问题**：Adam 中 Heavy Ball 动量的滞后超调。

**六维评估变化**：收敛速度略优于 Adam，山谷震荡抑制略优，其余维度基本持平。

---

### RAdam：动态监控二阶矩方差

> 由 Liu 等人于2019年提出。

**设计思路**：动态监控二阶矩估计的方差。早期方差过大时退化为基础更新；稳定后切换到 Adam。

**针对的问题**：Adam 在训练初期二阶矩估计不准确，可能导致异常更新。

**六维评估变化**：超参数鲁棒性更好（早期更稳定），收敛速度略慢（早期保守），其余持平。

---

### AdamW：解耦权重衰减

> 由 Loshchilov 和 Hutter 于2017年提出。

**设计思路**：将 L2 正则化（权重衰减）从 Adam 的梯度更新中**解耦**——直接在参数更新后应用权重衰减。

**改进效果**：解耦后的权重衰减在自适应学习率下表现更稳定，正则化效果更可控。

## 2.9 独立设计：ASGD（Averaged SGD）

> 由 Polyak 和 Juditsky 于1992年提出。

### 设计思路

ASGD **不改变更新规则本身**，而是在更新轨迹上做**后处理平滑**——这是一个独立的、可叠加的策略。

### 迭代规则

底层使用基础更新：

$$
\theta_{t+1} = \theta_t - \eta \cdot g_t
$$

额外维护参数的历史平均：

$$
\bar{\theta}_t = \frac{1}{t} \sum_{k=1}^{t} \theta_k
$$

最终推理时使用 $\bar{\theta}$ 而不是 $\theta_t$。

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | 依赖底层算法 | 不改变更新规则 |
| **山谷震荡抑制** | 依赖底层算法 | 不改变底层特性 |
| **逃离鞍点** | 依赖底层算法 | 不改变底层特性 |
| **超参数鲁棒性** | 依赖底层算法 | 额外引入平均窗口参数 |
| **显存开销** | 低-中 | 额外维护一份参数平均值 |
| **尺度不变性** | 依赖底层算法 | 不改变底层特性 |

### 定位

ASGD 是一个**独立的补充策略**，可叠加在任何基础优化器之上。

## 2.10 独立设计：Muon（大模型时代的新方向）

> **定位**：Muon 是一个**独立设计**的优化器，专为神经网络中的**2D权重矩阵**（如 Transformer 的注意力层和 FFN 层）设计，不属于 Adam 演进链条。

### 设计思路

1. **累积动量**：对梯度进行标准的累积，形成动量矩阵。
2. **正交化**：通过高效的**牛顿-舒尔茨迭代（Newton-Schulz iteration）**，将动量矩阵"掰正"为**最近的正交矩阵**。

**核心机制**：AdamW 按元素缩放，而 Muon 从**全局视角**重新调整整个权重矩阵的更新方向。

### 六维评估

| 维度 | 表现 | 原因 |
|------|------|------|
| **收敛速度** | **极快** | 正交化改善更新方向，数据效率高 |
| **山谷震荡抑制** | **好** | 正交化使更新方向更稳定 |
| **逃离鞍点** | **强** | 全局方向调整，摆脱梯度幅值束缚 |
| **超参数鲁棒性** | 中 | 需针对 2D 矩阵调参，非 2D 参数需配合其他优化器 |
| **显存开销** | 中 | 只维护一个动量缓冲区，约为 Adam 的一半 |
| **尺度不变性** | **好**（针对2D矩阵） | 正交化使更新方向与梯度幅值解耦 |

### 权衡

- **适用参数有限**：专为 2D 矩阵设计，不适用于偏置、LayerNorm、嵌入层等
- **神经元饥饿**：正交化机制可能导致 MLP 层部分神经元"死亡"

### 生产级应用

PyTorch 官方已宣布 **DeepSpeed 支持 Muon**。月之暗面（Kimi-K2）、智谱AI（GLM-4.5/GLM-5）、深度求索（DeepSeek-V4）等已在生产中采用。

# 第三部分：完整演进逻辑图

```
基础迭代框架：沿负梯度方向走一步
    Δθ = -η·g(θ)  （GD）
    │
    ├── 问题1：山谷中锯齿震荡，收敛慢
    │       ↓
    │   设计1：重球法（Heavy Ball Method / Momentum）
    │       ├── 维护历史速度的指数移动平均，抑制垂直震荡，加速水平推进
    │       └── 权衡：动量滞后，急转弯时超调
    │           ↓
    │       设计2：Nesterov Accelerated Gradient（NAG）
    │           ├── 先按历史惯性往前看一步，在该位置计算梯度，缓解超调
    │           └── 权衡：所有参数共用学习率，无法适应尺度差异
    │
    ├── 问题2：统一学习率无法适配不同参数
    │       ↓
    │   设计3：Rprop
    │       ├── 用梯度符号决定方向，每个参数独立维护步长大小
    │       └── 局限：依赖确定性梯度，噪声下符号频繁抖动
    │           ↓
    │       设计4：Adagrad
    │           ├── 用历史梯度平方的累加和缩放学习率，稀疏参数大步更新
    │           └── 权衡：累加和单调递增 → 训练后期强制停滞
    │               ↓
    │           设计5：RMSprop / Adadelta
    │               ├── 梯度平方的指数移动平均，解决了学习率单调递减问题
    │               └── 权衡：缺少一阶动量加速，平滑地形收敛慢
    │
    └── 问题3：需要同时具备加速和自适应
            ↓
        设计6：Adam
            ├── 一阶动量（Heavy Ball）+ 二阶矩自适应（RMSprop）+ 偏差修正
            ├── 权衡：动量滞后 → NAdam；早期不稳定 → RAdam
            └── AdamW：权重衰减与梯度更新解耦，大模型首选
                ├── NAdam：Nesterov预判 + Adam
                └── RAdam：动态监控早期二阶矩方差

独立设计（非演进链）：
    ├── ASGD（Averaged SGD）：参数历史平均，可叠加在任何优化器之上
    └── Muon：正交化动量，专为2D矩阵设计，大模型时代新方向
```

# 第四部分：各算法的决策规则一览

| 算法 | 历史信息维护 | 下一步决策规则 | 针对的问题 | 引入的权衡 |
|------|-------------|---------------|-----------|----------|
| **GD** | 无 | Δθ = −η·g | 无（基线） | 山谷震荡；统一学习率 |
| **Heavy Ball（Momentum）** | 一阶动量 EMA | v = βv − η·g, Δθ = v | 山谷震荡 | 动量滞后超调 |
| **NAG** | 一阶动量 EMA | v = βv − η·g(θ+βv), Δθ = v | 动量滞后超调 | 统一学习率 |
| **Rprop** | 参数独立步长 | Δθᵢ = −sign(gᵢ)·Δᵢ | 统一学习率 | 依赖确定性梯度 |
| **Adagrad** | 梯度平方累加和 | Δθᵢ = −η·gᵢ/√(Σg²+ε) | 稀疏梯度 + 统一学习率 | 训练后期停滞 |
| **RMSprop** | 梯度平方 EMA | Δθᵢ = −η·gᵢ/√(EMA(g²)+ε) | 训练停滞 | 缺少动量加速 |
| **Adam** | 一阶动量 EMA + 二阶矩 EMA | Δθ = −η·m̂/√(v̂+ε) | 加速 + 自适应 | 高显存 |
| **NAdam** | 一阶动量 EMA + 二阶矩 EMA | 在 Adam 上叠加 Nesterov 预判 | 动量滞后 | 对超参数敏感 |
| **RAdam** | 一阶动量 EMA + 二阶矩 EMA | 早期退化，稳定后 Adam | 早期不稳定性 | 仍存泛化短板 |
| **AdamW** | 一阶动量 EMA + 二阶矩 EMA | 解耦权重衰减的 Adam | 正则化稳定性 | 仍有 Adam 的显存开销 |
| **ASGD** | 参数历史平均 | 基础更新 + θ̄ = (1/t)Σθ | 参数震荡抑制 | 推理时需用平均值 |
| **Muon** | 动量矩阵 + 正交化 | 动量累积 → 牛顿-舒尔茨正交化 → 更新 | 2D矩阵全局方向 | 仅适用2D矩阵；神经元饥饿 |

# 第五部分：大模型时代的生产级选择

在大规模预训练和微调的实际生产中，优化器的选择已经形成了相对成熟的经验法则。

## 5.1 主流选择：AdamW

**AdamW** 是当前大模型训练中**适用范围最广**的优化器。它兼具优秀的收敛速度、超参数鲁棒性和尺度不变性，尽管显存开销高，但在大模型场景下其综合表现仍然最优。

### 典型配置（LLM 训练）

| 超参数 | 推荐值 | 说明 |
|--------|--------|------|
| betas | (0.9, 0.95) | 比默认 (0.9, 0.999) 在 bf16 下更稳定 |
| weight_decay | 0.1 ~ 0.2 | 对 2D 权重应用，不对偏置/LayerNorm/嵌入层应用 |
| 学习率调度 | 预热 + 余弦退火 | Transformer 训练的"标准配方" |
| 精度 | bfloat16 | 动态范围大，不易溢出 |

## 5.2 新兴选择：Muon

**核心差异**：AdamW 按元素缩放，Muon 用正交化从全局调整更新方向。显存占用约为 AdamW 的一半，数据效率更高。

**生产级应用**：PyTorch 官方已宣布 DeepSpeed 支持 Muon。月之暗面（Kimi-K2）、智谱AI（GLM-4.5/GLM-5）、深度求索（DeepSeek-V4）等已采用。

## 5.3 工程优化技术（可叠加）

| 技术 | 效果 | 说明 |
|------|------|------|
| **8-bit 优化器** | 显存减少约 75% | 使用 `bitsandbytes` 库 |
| **混合精度训练** | 显存减半、计算加速 | 广泛采用 bfloat16 |
| **参数分组** | 训练稳定性 | 对不同参数应用不同 weight_decay |

## 5.4 选择建议

| 场景 | 推荐 | 核心理由 |
|------|------|---------|
| 通用大模型预训练 | AdamW | 收敛快 + 超参数鲁棒 + 工具链成熟 |
| 追求极致显存效率 | Muon + 8-bit | 显存中 + 收敛极快 |
| 小规模实验/原型验证 | AdamW | 超参数健壮 |
| 显存极度受限 | GD | 显存极低 |

# 第六部分：从损失函数视角判断算法可行性

## 6.1 理论基础

对于一个确定的神经网络，其损失函数 $L(\theta)$ 是一个确定的复合函数，其数学表达式完全已知。理论上，我们可以写出它的梯度和 Hessian 表达式。

## 6.2 从函数性质可以推导出的结论

| 函数性质 | 对算法选择的理论提示 |
|---------|-------------------|
| 网络包含非线性激活 → 损失函数**非凸** | 任何依赖凸性保证的算法都没有全局最优保证 |
| ReLU 激活 → 损失函数**分段线性** | 梯度在大部分区域为常数，Rprop 符号依赖算法理论上有效 |
| Sigmoid/Tanh → 存在**饱和区** | 梯度消失 → 需要自适应学习率（Adam 家族）补偿 |
| 损失曲面条件数大（病态） | 需要动量或自适应方法来缓解震荡 |
| 梯度具有稀疏性 | Adagrad/Adam 的独立步长机制理论上更优 |

## 6.3 实践中能做到 vs 不能做到的

| 能做到的（诊断层面） | 做不到的（全局预判层面） |
|-------------------|----------------------|
| 估计当前点的 Hessian 谱 | 无法测绘高维全局地形 |
| 检测当前点的病态程度 | 无法由局部信息推断全局最优位置 |
| 判断训练是否陷入尖锐区域 | 无法预知哪个优化器最终表现最好 |
| 监测梯度噪声水平 | 无法在训练前推导出最优超参数 |

**结论**：理论分析可以用来**排除**明显不合适的算法，但最终选择需要靠**经验法则 + 小规模实验**来确认。

# 第七部分：算法选择的三层决策框架

```
第1层：理论排除（基于损失函数的已知性质）
    ├── 非凸 → 排除仅适用于凸优化的方法
    ├── 梯度带噪声 → 排除Rprop（纯理论中无噪声）
    ├── 梯度稀疏 → 倾向Adagrad/Adam
    ├── 存在梯度消失 → 倾向自适应方法
    └── 病态曲率 → 需要动量或自适应

第2层：经验法则（基于六维评估 + 任务类型）
    ├── 追求极简可解释性 → GD
    ├── NLP / Transformer（大模型预训练） → AdamW（首选）/ Muon（前沿探索）
    ├── 稀疏特征 / 推荐系统 → Adagrad/Adam
    ├── 快速原型 / 不确定 → Adam（鲁棒性好）
    └── 显存受限的大模型 → Muon 或 8-bit AdamW

第3层：实验验证
    ├── 候选2-3个优化器（如 AdamW vs Muon vs GD）
    ├── 小规模验证集对比：收敛速度 + 验证精度 + 显存占用
    └── 最终确定
```

## 关键认知

1. **没有普适最优算法**——每个算法针对特定问题设计，同时引入新的权衡
2. **Heavy Ball（Momentum）不解决局部最优**——它只抑制山谷震荡
3. **自适应 ≠ 自动调参**——基础学习率等超参数仍需调试
4. **确定性框架下，任何一阶方法都无法保证逃离局部极小值**——这是非凸优化的根本限制
5. **逃离鞍点 ≠ 逃离局部极小值**——鞍点处有方向信息可利用，局部极小值处所有方向都是上升的
6. **理论可以排除，不能预选**——最终选择靠实验验证
7. **AdamW 是大模型时代的通用基石**——综合表现最优
8. **Muon 是独立设计的新方向**——专为 2D 矩阵优化，效率更高

# 附录：术语对照

| 学术/数学名称 | 深度学习社区常用名称 | 提出者/年份 |
|-------------|-------------------|-----------|
| **重球法（Heavy Ball Method）** | Momentum（动量法） | Polyak, 1964 |
| **Nesterov Accelerated Gradient（NAG）** | Nesterov Momentum | Nesterov, 1983 |
| **Rprop** | Rprop | Braun & Riedmiller, 1993 |
| **Adagrad** | Adagrad | Duchi et al., 2011 |
| **RMSprop** | RMSprop | Hinton, 2012（未正式发表） |
| **Adadelta** | Adadelta | Zeiler, 2012 |
| **Adam** | Adam | Kingma & Ba, 2014 |
| **AdamW** | AdamW | Loshchilov & Hutter, 2017 |
| **NAdam** | NAdam | Dozat, 2016 |
| **RAdam** | RAdam | Liu et al., 2019 |
| **ASGD（Averaged SGD）** | ASGD | Polyak & Juditsky, 1992 |
| **Muon（MomentUm Orthogonalized by Newton-schulz）** | Muon | FAIR, 2024（独立设计） |

> **说明**：深度学习社区习惯用 "Momentum" 这个直观的物理比喻来称呼重球法，但在学术文献和优化理论中，"重球法（Heavy Ball Method）" 是更准确的名称。本文在正文中优先使用学术名称，同时在括号中标注社区常用名。

> **关于 Muon 的定位**：Muon 是一个**独立设计**的优化器，专为 2D 权重矩阵优化而生，不属于从 GD 到 Adam 的演进链条。

# 附录：PyTorch 代码对应

```python
import torch
from torch import optim

net = torch.nn.Linear(16, 2)

# ── 基础梯度下降 ──
optim.SGD(net.parameters(), lr=0.01)                      # GD

# ── 动量类（Heavy Ball / NAG） ──
optim.SGD(net.parameters(), lr=0.01, momentum=0.9)        # Heavy Ball（Momentum）
optim.SGD(net.parameters(), lr=0.01, momentum=0.9, nesterov=True)  # NAG

# ── 自适应步长策略 ──
optim.Rprop(net.parameters(), lr=0.01)                    # Rprop
optim.Adagrad(net.parameters(), lr=0.01)                  # Adagrad
optim.RMSprop(net.parameters(), lr=0.01)                  # RMSprop
optim.Adadelta(net.parameters())                          # Adadelta

# ── Adam 家族 ──
optim.Adam(net.parameters(), lr=1e-3)                     # Adam
optim.AdamW(net.parameters(), lr=1e-3)                    # AdamW
optim.NAdam(net.parameters(), lr=1e-3)                    # NAdam
optim.RAdam(net.parameters(), lr=1e-3)                    # RAdam

# ── 独立支线 ──
optim.ASGD(net.parameters(), lr=0.01)                     # ASGD

# ── 大模型时代工程优化（需 bitsandbytes） ──
# import bitsandbytes as bnb
# bnb.optim.AdamW8bit(net.parameters(), lr=1e-3)           # 8-bit AdamW

# ── Muon（需独立安装） ──
# from muon import Muon
# optim.Muon(net.parameters(), lr=3e-3, momentum=0.95)
```

# 关键概念澄清

| 常见误解 | 正确理解 |
|---------|---------|
| Adam 是最先进的优化器 | Adam 是**适用范围最广的之一**；在大模型时代，AdamW 是通用基石，Muon 是独立设计的新方向 |
| 新算法一定优于旧算法 | 每个算法针对特定问题设计，同时引入新的**权衡**，无绝对优劣 |
| Momentum 帮助跳出局部最优 | 重球法（Momentum）只**抑制山谷震荡**，不解决局部最优 |
| 自适应学习率 = 不需要调参 | 基础学习率等超参数**仍需调试** |
| Rprop 可以用于所有场景 | Rprop **仅适用于确定性梯度**，噪声下符号频繁抖动 |
| Muon 是 Adam 的替代品/改进版 | Muon 是**独立设计**，专为 2D 权重矩阵优化，与 AdamW 并行而非继承 |
| 8-bit 优化器是一种新算法 | 8-bit 优化器是一种**工程优化技术**，可叠加在任何优化器之上 |
| 理论适用的算法一定能用 | 理论给出的是**充分条件**，超出区间仍可能有效 |
| Momentum 和重球法是两个不同的东西 | 它们是**同一个算法**，学术名为重球法，社区俗称动量法 |
| 梯度噪声 = 算法本身的问题 | 梯度噪声来源于**数据采样方式**，与参数更新策略正交 |
| Adam 一定能逃离局部极小值 | **在确定性框架下，Adam 也无法保证逃离局部极小值**——所有一阶方法都只能收敛到初始点吸引域内的某个局部极小值 |
| 逃离鞍点 = 逃离局部极小值 | **不是。** 鞍点处有方向信息可利用；局部极小值处所有方向都是上升的，算法无信息可判断往哪走 |
| 评估算法只需看收敛曲线 | 需综合评估**收敛速度、山谷震荡抑制、逃离鞍点、鲁棒性、显存、尺度不变性**六维指标（见第一部分） |